In [ ]:
import polars as pl

In [2]:
df = pl.read_parquet("../data/parquet/*.parquet")
df

white_elo,black_elo,result,moves,opening
i32,i32,i8,str,str
2071,2030,1,"""e2e4 e7e5 g1f3 d7d6 d2d4 e5d4 …","""Philidor Defense: Exchange Var…"
2098,2152,1,"""e2e4 e7e5 g1f3 g8f6 b1c3 b8c6 …","""Four Knights Game: Scotch Vari…"
2080,2180,-1,"""e2e4 e7e5 g1f3 g8f6 b1c3 f8b4 …","""Russian Game: Three Knights Ga…"
2072,2067,1,"""e2e4 c7c5 g1f3 d7d6 d2d4 c5d4 …","""Sicilian Defense: Najdorf Vari…"
2091,2001,-1,"""g2g4 d7d5 f1g2 c7c6 h2h3 e7e5 …","""Grob Opening: Keene Defense"""
…,…,…,…,…
2131,2133,-1,"""e2e4 c7c5 g1f3 e7e6 d2d4 c5d4 …","""Sicilian Defense: Kan Variatio…"
2171,2112,1,"""e2e4 g7g6 d2d4 f8g7 b1c3 a7a6 …","""Modern Defense: Standard Line"""
2072,2290,1,"""g1f3 g7g6 d2d4 f8g7 c2c4 b7b6 …","""Zukertort Opening: Kingside Fi…"


In [3]:
# opening is whatever is before :, before , and without any #2, #3, etc. suffixes
# normalize whitespace so e.g. 'Vienna Game' and 'Vienna Game ' dedupe correctly
openings = (
    df["opening"]
    .str.split(":")
    .list.get(0)
    .str.split(",")
    .list.get(0)
    .str.replace(r"#\d+", "")
    .str.strip_chars()
    .str.replace_all(r"\s+", " ")
    .unique()
    .sort()
)

for opening in openings:
    print(opening)

Alekhine Defense
Amar Opening
Amazon Attack
Anderssen Opening
Anderssen's Opening
Australian Defense
Barnes Defense
Barnes Opening
Benko Gambit
Benko Gambit Accepted
Benko Gambit Declined
Benoni Defense
Bird Opening
Bishop's Opening
Blackmar-Diemer Gambit
Blackmar-Diemer Gambit Accepted
Blackmar-Diemer Gambit Declined
Blumenfeld Countergambit
Blumenfeld Countergambit Accepted
Bogo-Indian Defense
Borg Defense
Budapest Defense
Canard Opening
Caro-Kann Defense
Carr Defense
Catalan Opening
Center Game
Center Game Accepted
Clemenz Opening
Colle System
Czech Defense
Danish Gambit
Danish Gambit Accepted
Danish Gambit Declined
Duras Gambit
Dutch Defense
East Indian Defense
Elephant Gambit
English Defense
English Opening
English Orangutan
Englund Gambit
Englund Gambit Complex
Englund Gambit Complex Declined
Englund Gambit Declined
Four Knights
Four Knights Game
Franco-Benoni Defense
French Defense
Gedult's Opening
Giuoco Piano
Goldsmith Defense
Grob Opening
Gruenfeld Defense
Grünfeld Defense
Gu

In [4]:
moves = df[0]["moves"].to_list()[0]
print(moves)

e2e4 e7e5 g1f3 d7d6 d2d4 e5d4 f3d4 f8e7 f1c4 g8f6 b1c3 e8g8 e1g1 a7a6 a2a3 c7c5 d4e2 b8c6 f2f4 c8g4 d1e1 b7b5 c4d5 f6d5 c3d5 c6d4 e2d4 c5d4 f4f5 f7f6 c1f4 a8c8 e1d2 g4h5 a1c1 h5f7 d2d4 c8c4 d4d3 f7d5 d3d5 g8h8 c2c3 d8b6 g1h1 a6a5 c1d1 c4c6 d5e6 b5b4 e6e7 f8d8 c3b4 a5b4 a3b4 h7h6 b4b5 c6c8 d1d6 d8d6 f4d6 b6b5 f1e1 c8e8 e7c7 e8e4 e1c1 b5f5 h2h3 e4e2 b2b4 f5f2 c7c8 h8h7 c8g4 e2b2 d6c5 f2d2 c1d1 d2c2 d1e1 f6f5 g4g3


In [5]:
moves = df["moves"].str.split(" ")
moves

moves
list[str]
"[""e2e4"", ""e7e5"", … ""g4g3""]"
"[""e2e4"", ""e7e5"", … ""e5d6""]"
"[""e2e4"", ""e7e5"", … ""c2g2""]"
"[""e2e4"", ""c7c5"", … ""e1e4""]"
"[""g2g4"", ""d7d5"", … ""f2f1""]"
…
"[""e2e4"", ""c7c5"", … ""g5b5""]"
"[""e2e4"", ""g7g6"", … ""d3g6""]"
"[""g1f3"", ""g7g6"", … ""f7d7""]"


In [6]:
all_moves = moves.explode().unique().sort()  # just experimenting, they are not all possible moves
all_moves

moves
str
"""a1a2"""
"""a1a3"""
"""a1a4"""
"""a1a5"""
"""a1a6"""
…
"""h8h3"""
"""h8h4"""
"""h8h5"""


In [7]:
with open("../data/all_uci_moves.txt") as f:
    all_uci_moves = [line.strip() for line in f]

all_uci_moves

['a1h8',
 'a1a8',
 'a1g7',
 'a1a7',
 'a1f6',
 'a1a6',
 'a1e5',
 'a1a5',
 'a1d4',
 'a1a4',
 'a1c3',
 'a1a3',
 'a1b2',
 'a1a2',
 'a1h1',
 'a1g1',
 'a1f1',
 'a1e1',
 'a1d1',
 'a1c1',
 'a1b1',
 'a2g8',
 'a2a8',
 'a2f7',
 'a2a7',
 'a2e6',
 'a2a6',
 'a2d5',
 'a2a5',
 'a2c4',
 'a2a4',
 'a2b3',
 'a2a3',
 'a2h2',
 'a2g2',
 'a2f2',
 'a2e2',
 'a2d2',
 'a2c2',
 'a2b2',
 'a2b1',
 'a2a1',
 'a3f8',
 'a3a8',
 'a3e7',
 'a3a7',
 'a3d6',
 'a3a6',
 'a3c5',
 'a3a5',
 'a3b4',
 'a3a4',
 'a3h3',
 'a3g3',
 'a3f3',
 'a3e3',
 'a3d3',
 'a3c3',
 'a3b3',
 'a3b2',
 'a3a2',
 'a3c1',
 'a3a1',
 'a4e8',
 'a4a8',
 'a4d7',
 'a4a7',
 'a4c6',
 'a4a6',
 'a4b5',
 'a4a5',
 'a4h4',
 'a4g4',
 'a4f4',
 'a4e4',
 'a4d4',
 'a4c4',
 'a4b4',
 'a4b3',
 'a4a3',
 'a4c2',
 'a4a2',
 'a4d1',
 'a4a1',
 'a5d8',
 'a5a8',
 'a5c7',
 'a5a7',
 'a5b6',
 'a5a6',
 'a5h5',
 'a5g5',
 'a5f5',
 'a5e5',
 'a5d5',
 'a5c5',
 'a5b5',
 'a5b4',
 'a5a4',
 'a5c3',
 'a5a3',
 'a5d2',
 'a5a2',
 'a5e1',
 'a5a1',
 'a6c8',
 'a6a8',
 'a6b7',
 'a6a7',
 'a6h6',
 'a6g6',
 

In [8]:
tokenizer = {move: idx for idx, move in enumerate(all_uci_moves, start=3)}
tokenizer["<SOS>"] = 0
tokenizer["<EOS>"] = 1
tokenizer["<PAD>"] = 2
tokenizer

{'a1h8': 3,
 'a1a8': 4,
 'a1g7': 5,
 'a1a7': 6,
 'a1f6': 7,
 'a1a6': 8,
 'a1e5': 9,
 'a1a5': 10,
 'a1d4': 11,
 'a1a4': 12,
 'a1c3': 13,
 'a1a3': 14,
 'a1b2': 15,
 'a1a2': 16,
 'a1h1': 17,
 'a1g1': 18,
 'a1f1': 19,
 'a1e1': 20,
 'a1d1': 21,
 'a1c1': 22,
 'a1b1': 23,
 'a2g8': 24,
 'a2a8': 25,
 'a2f7': 26,
 'a2a7': 27,
 'a2e6': 28,
 'a2a6': 29,
 'a2d5': 30,
 'a2a5': 31,
 'a2c4': 32,
 'a2a4': 33,
 'a2b3': 34,
 'a2a3': 35,
 'a2h2': 36,
 'a2g2': 37,
 'a2f2': 38,
 'a2e2': 39,
 'a2d2': 40,
 'a2c2': 41,
 'a2b2': 42,
 'a2b1': 43,
 'a2a1': 44,
 'a3f8': 45,
 'a3a8': 46,
 'a3e7': 47,
 'a3a7': 48,
 'a3d6': 49,
 'a3a6': 50,
 'a3c5': 51,
 'a3a5': 52,
 'a3b4': 53,
 'a3a4': 54,
 'a3h3': 55,
 'a3g3': 56,
 'a3f3': 57,
 'a3e3': 58,
 'a3d3': 59,
 'a3c3': 60,
 'a3b3': 61,
 'a3b2': 62,
 'a3a2': 63,
 'a3c1': 64,
 'a3a1': 65,
 'a4e8': 66,
 'a4a8': 67,
 'a4d7': 68,
 'a4a7': 69,
 'a4c6': 70,
 'a4a6': 71,
 'a4b5': 72,
 'a4a5': 73,
 'a4h4': 74,
 'a4g4': 75,
 'a4f4': 76,
 'a4e4': 77,
 'a4d4': 78,
 'a4c4': 79,
 'a4b4

In [9]:
df2 = pl.read_parquet("../data/tokenized_games.parquet")
df2

token_ids
list[u16]
"[0, 760, … 1]"
"[0, 760, … 1]"
"[0, 760, … 1]"
"[0, 760, … 1]"
"[0, 1140, … 1]"
…
"[0, 760, … 1]"
"[0, 760, … 1]"
"[0, 1732, … 1]"


In [10]:
# length of longest game
max_length = df2["token_ids"].list.len().max()
max_length

390

In [17]:
max_length_og_id = df["moves"].str.split(" ").list.len().arg_max()
max_length_og = df[max_length_og_id]["moves"].item()
print(max_length_og)
print(len(max_length_og.split(" ")))

d2d4 g8f6 g1f3 g7g6 c2c4 f8g7 b1c3 e8g8 e2e4 d7d6 f1e2 e7e5 e1g1 e5d4 f3d4 f8e8 f2f3 b8c6 c1e3 c6d4 e3d4 g7h6 d1c2 c7c6 a1d1 d8e7 e2d3 a7a6 a2a4 c8e6 f3f4 f6g4 d3e2 a8c8 c2d3 e7h4 h2h3 c6c5 d4c5 c8c5 e2g4 e6c4 d3d6 h6f8 d6d7 c5c8 g4e2 c4e6 d7d3 e6b3 d1a1 e8d8 c3d5 b3d5 e4d5 c8c5 a1d1 c5d5 d3d5 d8d5 d1d5 f8h6 d5d7 h6f4 d7b7 h4g3 b2b3 g3h2 g1f2 h2g3 f2g1 g3h2 g1f2 g8f8 b7d7 h2g3 f2g1 g3h2 g1f2 f8e8 d7b7 h2g3 f2g1 g3h2 g1f2 f4g3 f2e3 g3c7 e2f3 h2e5 e3d3 e5f5 d3c3 c7e5 c3d2 e5f4 d2c3 f4e5 c3d2 e8f8 f1d1 e5d4 d1e1 d4f6 e1d1 f5e5 d2d3 h7h5 d1d2 f8g7 d3c2 e5c3 c2d1 f6d4 d2c2 c3d3 c2d2 d3f1 d1c2 f1a1 d2d3 a1b2 c2d1 d4c3 f3d5 g7h6 b7f7 c3b4 d5c4 h5h4 f7f4 b2b1 d1e2 b1e1 e2f3 e1g3 f3e4 g3g2 f4f3 g2g5 d3d5 g5e7 d5e5 e7b7 e5d5 b7e7 d5e5 e7b7 e4e3 a6a5 e5e4 b4d2 e3d4 b7g7 d4d3 d2g5 d3e2 g7b2 e2f1 g5f6 e4g4 b2e5 f1g2 g6g5 c4d3 e5b2 g2f1 b2a1 f1e2 f6c3 e2f2 c3d4 f2e2 d4c3 e2f2 a1h1 g4c4 h1h2 f2f1 h2h1 f1f2 c3e1 f2e3 h1g1 e3e4 g1b6 e4d5 b6d8 d5e4 d8b6 e4d5 b6d8 d5e4 d8e7 e4d4 e7a7 d4e4 a7e7 e4d4 e7d6 